1번 문항
llm 대화 링크가 생성되지 않아서 맨 아래에 대화내역 첨부하였습니다.

In [16]:
import csv
import json
import logging
import re

# 로그 설정
logger = logging.getLogger()
if logger.hasHandlers():
    logger.handlers.clear()
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def make_report(csv_path: str, json_path: str) -> int:
    try:
        #데이터 읽기
        with open(csv_path, "r", encoding='utf-8') as f:
            reader = csv.DictReader(f)
            rows = list(reader)

        new_data = []

        #None처리와 중간 기말 과제 성적 저장
        for row in rows:
            mid = int(row['중간']) if row.get('중간') else None
            final = int(row['기말']) if row.get('기말') else None
            hw = int(row['과제']) if row.get('과제') else None
            
            w_sum = None
            grade = None
            

            # 가중평균 계산
            if None not in (mid, final, hw):
                w_sum = round(0.3 * mid + 0.5 * final + 0.2 * hw, 1)
                if w_sum >= 90: grade = 'A'
                elif w_sum >= 80: grade = 'B'
                elif w_sum >= 70: grade = 'C'
                else: grade = 'F'

            student_info = {
                "이름": row['이름'],
                "학번": row['학번'],
                "점수": {"중간": mid, "기말": final, "과제": hw},
                "평균": w_sum,
                "등급": grade
            }
            new_data.append(student_info)
            
            # 로그 출력
            logging.info(f"{student_info['이름']}: {student_info['평균']}, {student_info['등급']}")

        # [JSON 저장
        with open(json_path, "w", encoding='utf-8') as f:
            json_str = json.dumps(new_data, ensure_ascii=False, indent=4)
            # 점수 부분만 한 줄로 합치기
            compact_json = re.sub(r'("점수": \{)\s+(.*?)\s+(\})', 
                                  lambda m: m.group(1) + " ".join(m.group(2).split()) + m.group(3), 
                                  json_str, flags=re.DOTALL)
            f.write(compact_json)

        return len(new_data)

    except FileNotFoundError:
        logging.warning(f"파일 {csv_path}을 찾을 수 없습니다.")
        return 0
    except UnicodeDecodeError:
        logging.error(f"파일 {csv_path}의 인코딩이 잘못되었습니다")
    except Exception as e:
        logging.error(f"오류 발생: {e}")
        return 0



In [17]:
make_report('scores.csv','report.json')


INFO: 김언어: 89.5, B
INFO: 이국문: 84.4, B
INFO: 박영문: 93.5, A
INFO: 최역사: None, None


4

In [18]:
make_report('missing.csv','report.json')


0

출력:
>>>make_report('missing.csv','report.json')
WARNING: 파일 missing.csv을 찾을 수 없습니다.
0
>>>make_report('scores.csv','report.json')
INFO: 김언어: 89.5, B
INFO: 이국문: 84.4, B
INFO: 박영문: 93.5, A
INFO: 최역사: None, None
4

report.json 파일:
[
    {
        "이름": "김언어",
        "학번": "2026-10000",
        "점수": {"중간": 85, "기말": 92, "과제": 90},
        "평균": 89.5,
        "등급": "B"
    },
    {
        "이름": "이국문",
        "학번": "2026-12345",
        "점수": {"중간": 78, "기말": 88, "과제": 85},
        "평균": 84.4,
        "등급": "B"
    },
    {
        "이름": "박영문",
        "학번": "2026-13579",
        "점수": {"중간": 95, "기말": 90, "과제": 100},
        "평균": 93.5,
        "등급": "A"
    },
    {
        "이름": "최역사",
        "학번": "2025-11111",
        "점수": {"중간": null, "기말": 82, "과제": 88},
        "평균": null,
        "등급": null
    }
]

1번 코드 설명:

로그 핸들러의 중복을 방지하기 위해 logger.hanlders.clear()를 활용하였다.

INFO 로그를 출력하기 위해 basicConfig(level=logging.INFO) 로 설정하였다

함수설명:
open함수를 이용하여 데이터를 리스트로 읽어온다.
json파일로 출력할 새 데이터를 저장할 새 리스트를 만든다.

삼항연산자를 이용해서 중간 기말 과제에 대한 값을 저장한다. 이때 None값에 대해서도 처리를 한다. 이때 .get()은 key가 존재하면 값을 반환하고 없으면 None을 반환한다.

조건문을 이용하여 등급을 산출하고 모든 정보를 student_info 딕셔너리에 저장한다. 모든 student_info를 new_data에 저장한다.


json.dump를 json파일을 쓴다. 이때 점수 부분의 출력을 과제의 예시에 맞추기 위해서 re모듈을 이용하여 바꾸어준다. 정규표현식은 json 문자열 내에서 "점수": {...} 형태를 찾아 세 그룹으로 나눈다. \s+로 나타나는 공백과 줄바꿈을 제거하여 문제에서 나타난 형식으로 바꾼 뒤 세 그룹을 다시 합쳤다.

Exception을 통해 에러가 발생했을 때 로그를 띄운다.

인코딩은 한글 인코딩의 표준인 Utf-8을 선택하였다.

2번문항

In [1]:
#a

class InvalidJamoError(ValueError):
    pass

Exception을 통해 정의할 경우 너무나도 넓은 범위의 오류를 포괄하여 정확히 어떤 종류의 오류가 발생했는지 알기 어렵다. 그에비해 ValueError의 자식클래스로 정의할 경우 구체적으로 값이 조건에 맞지 않음을 알 수 있다.

In [14]:
def classify_jamo(c: str) ->str:
    if type(c) != str:
        raise TypeError(f"문자열이 아닙니다:{c}")
    elif len(c) != 1:
        raise ValueError(f"길이가 1이 아닙니다: {c}")
    elif 0x3131 <= ord(c) <= 0x314E:
        return "자음"
    elif 0x314F <= ord(c) <= 0x3163:
        return "모음"
    else:
        raise InvalidJamoError(f"한글 자모가 아닙니다: {c}")
    

In [15]:
inputs = ["ㄱ", 'ㅏ', 'ㄲ', "가", "AB", 5, "ㅎ", "ㅣ", ""]

for c in inputs:
    try:
        result = classify_jamo(c)
        print(result)
    except InvalidJamoError as e:
        print(f"[InvalidJamoError] {e}")
    except ValueError as e:
        print(f"[ValueError] {e}")
    except TypeError as e:
        print(f"[TypeError] {e}")



자음
모음
자음
[InvalidJamoError] 한글 자모가 아닙니다: 가
[ValueError] 길이가 1이 아닙니다: AB
[TypeError] 문자열이 아닙니다:5
자음
모음
[ValueError] 길이가 1이 아닙니다: 


결과:
자음
모음
자음
[InvalidJamoError] 한글 자모가 아닙니다: 가
[ValueError] 길이가 1이 아닙니다: AB
[TypeError] 문자열이 아닙니다:5
자음
모음
[ValueError] 길이가 1이 아닙니다: 

LLM 대화 내역 첨부




csv dictreader에서 데이터에 빈칸을 none으로 처리하는 방법이 뭐야


`csv.DictReader`는 기본적으로 빈 칸을 빈 문자열(`''`)로 읽어옵니다. 이를 `None`으로 바꾸는 가장 깔끔한 방법 두 가지를 소개해 드릴게요.

---

### 1. 제너레이터 표현식 사용 (가장 추천)
데이터를 한 줄씩 읽을 때, 딕셔너리의 value가 `''`인 경우만 `None`으로 바꿔주는 로직을 넣는 방식입니다. 메모리 효율적이고 직관적입니다.

```python
import csv

with open('data.csv', 'r') as f:
    reader = csv.DictReader(f)
    # value가 빈 문자열이면 None으로, 아니면 그대로 유지
    data = [{k: (v if v != '' else None) for k, v in row.items()} for row in reader]

for row in data:
    print(row)
```

---

### 2. 별도의 변환 함수 정의
데이터 처리 로직이 복잡해질 것 같다면, 변환하는 부분을 함수로 빼서 `map`을 활용할 수도 있습니다.

```python
import csv

def nullify(row):
    for k, v in row.items():
        if v == '':
            row[k] = None
    return row

with open('data.csv', 'r') as f:
    reader = csv.DictReader(f)
    # 각 row에 대해 nullify 함수 적용
    clean_data = map(nullify, reader)
    
    for row in clean_data:
        print(row)
```

---

### 💡 팁: Pandas를 사용한다면?
만약 데이터 분석이 목적이라 **Pandas** 라이브러리를 사용하신다면, 별도의 설정 없이도 빈 칸을 자동으로 `NaN`(Not a Number, Python의 `None`과 유사한 역할)으로 처리해줍니다.

```python
import pandas as pd

df = pd.read_csv('data.csv')
# 이미 빈 칸은 NaN으로 처리되어 있음
```

**참고 사항:**
`csv.DictReader` 자체 옵션(parameter) 중에는 빈 문자열을 자동으로 `None`으로 변환해주는 기능은 아쉽게도 없습니다. 따라서 위와 같이 데이터를 읽어온 직후에 가공하는 단계가 필요합니다.

이거 줄바꿈없이 출력 못해?
네, 당연히 가능하죠! 지금 보여주신 데이터는 **JSON(JavaScript Object Notation)** 형식인데요. 보통 가독성을 위해 줄바꿈을 넣지만, 프로그램 간의 통신이나 용량을 줄여야 할 때는 한 줄로 붙여서 출력하기도 합니다.

사용하시는 언어에 따라 아래와 같은 방법으로 줄바꿈 없이(Compact 한 형식으로) 출력할 수 있습니다.

---

### 1. Python (파이썬)
`json.dumps()` 함수를 사용할 때 별도의 `indent` 옵션을 주지 않으면 기본적으로 한 줄로 출력됩니다.

```python
import json

data = {
    "점수": {
        "중간": 85,
        "기말": 92,
        "과제": 90
    }
}

# 한 줄로 출력
print(json.dumps(data, ensure_ascii=False))
```
**출력 결과:** `{"점수": {"중간": 85, "기말": 92, "과제": 90}}`

---

### 2. JavaScript (자바스크립트)
`JSON.stringify()` 메서드에서 두 번째, 세 번째 인자를 비워두면 한 줄로 반환됩니다.

```javascript
const data = {
  "점수": {
    "중간": 85,
    "기말": 92,
    "과제": 90
  }
};

// 한 줄로 출력
console.log(JSON.stringify(data));
```

---

### 3. 온라인 도구 활용
코딩 중이 아니라 단순히 텍스트를 바꾸고 싶으신 거라면, **"JSON Minifier"** 또는 **"JSON Compressor"**라고 검색해서 나오는 사이트에 복사/붙여넣기를 하면 바로 한 줄로 만들어줍니다.

---

### 요약하자면
지금 보고 계신 형태는 "Pretty Print"라고 해서 사람이 보기 편하게 만든 상태이고, 이걸 **"Minify(축소)"** 하면 원하시는 대로 줄바꿈 없는 텍스트를 얻으실 수 있습니다. 

혹시 특정 프로그래밍 언어에서 바로 적용해야 하는 상황인가요?
이런식으로 출력해야해서 들어쓰기가 없는건 안돼
아하, 어떤 느낌인지 바로 알겠습니다! 전체를 다 한 줄로 붙이는 게 아니라, **"큰 덩어리는 줄바꿈을 하되, 특정 부분(예: 점수 딕셔너리)만 한 줄로"** 만들고 싶으신 거군요.

보여주신 이미지는 아주 전형적인 **혼합형 포맷**인데, 아쉽게도 파이썬이나 자바스크립트의 기본 `json.dumps` 라이브러리는 "여기부터 여기까지는 한 줄로 써"라는 식의 세부 옵션을 제공하지 않습니다. (기본 라이브러리는 전부 줄을 바꾸거나, 전부 안 바꾸거나 둘 중 하나만 하거든요.)

이런 결과물을 얻으려면 보통 두 가지 방법을 씁니다.

---

### 1. 직접 문자열을 조작하는 방법 (가장 확실함)
전체적으로 `indent`를 주어 출력한 뒤, `점수` 부분만 정규표현식이나 `replace`를 이용해 한 줄로 합치는 방식입니다.

```python
import json
import re

data = [
    {
        "이름": "김언어",
        "학번": "2026-10000",
        "점수": {"중간": 85, "기말": 92, "과제": 90},
        "평균": 89.5,
        "등급": "B"
    }
]

# 1. 일단 전체를 예쁘게 출력함
json_str = json.dumps(data, indent=4, ensure_ascii=False)

# 2. 정규표현식으로 "점수": { ... } 내부의 줄바꿈과 공백을 제거
# (복잡해 보이지만 '점수' 뒤의 중괄호 내용을 한 줄로 붙이는 마법입니다)
compact_score = re.sub(r'("점수":\s\{)\s+([^}]+)\s+\}', 
                       lambda m: m.group(1) + re.sub(r'\s+', ' ', m.group(2).strip()) + "}", 
                       json_str)

print(compact_score)
```

---

### 2. `separators` 옵션 활용 (간접적인 방법)
`indent`를 쓰면서 `separators` 값을 조절하면 콤마(`,`)나 콜론(`:`) 뒤의 공백을 없앨 수 있지만, 원하시는 것처럼 특정 중괄호만 쏙 골라서 한 줄로 만들기는 어렵습니다.

### 3. 외부 라이브러리 사용
만약 이런 작업이 아주 많다면 `commentjson`이나 `more-itertools` 같은 라이브러리를 쓰기도 하지만, 보통은 위 1번처럼 **필요한 부분만 살짝 가공**하는 게 제일 빠르고 속 편합니다.

---

### 결론
보여주신 형태는 **"커스텀 포맷"**이라서 기본 `indent` 옵션만으로는 불가능합니다. 

* **방법 A:** 일단 한 줄로 쭉 뽑은 다음에 수동으로 `\n`을 넣는다.
* **방법 B:** 일단 `indent=4`로 뽑은 다음에 "점수" 섹션만 한 줄로 합친다. (위 코드 참고)

이 중에서 어떤 방식이 작업하시기에 더 편할 것 같나요? 구체적인 언어를 말씀해주시면 해당 언어에 맞는 코드를 짜드릴게요!



ipynb에서 로그를 어떻게봐



Jupyter Notebook(`.ipynb`) 환경에서는 로그를 확인하는 방법이 크게 두 가지 상황(셀 실행 결과 vs. 시스템 로그)으로 나뉩니다.

---

## 1. 코드 실행 로그 확인 (가장 일반적)
코드 내에서 `print()`나 `logging` 모듈을 사용했다면 해당 셀의 **출력창(Output)**에 바로 나타납니다.

### 💡 `logging` 모듈 사용 시 주의점
기본적으로 파이썬의 `logging`은 이미 Jupyter 커널에 의해 설정되어 있는 경우가 많아, 그냥 쓰면 출력이 안 될 때가 있습니다. 아래처럼 설정을 초기화하고 사용하세요.

```python
import logging

# 기존 핸들러 삭제 후 재설정
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

logging.info("이 로그는 셀 바로 아래에 출력됩니다.")
```

---

## 2. 셀 출력 결과가 너무 길어 보이지 않을 때
로그 양이 너무 많으면 VS Code나 Jupyter가 출력을 자동으로 생략하거나 스크롤 안에 숨겨버립니다.

* **VS Code:** 출력창 왼쪽의 **`...` (더 보기)** 아이콘을 클릭하거나, 왼쪽 끝의 여백을 더블 클릭해서 전체를 펼칠 수 있습니다.
* **Jupyter Lab/Notebook:** 출력 창 위에서 마우스 오른쪽 버튼을 누르고 **`Clear Outputs`**(삭제) 또는 **`Enable Scrolling for Outputs`**(스크롤 생성)를 설정할 수 있습니다.

---

## 3. Jupyter 서버 자체 로그 (시스템 로그)
코드가 아예 멈췄거나 커널이 죽는 등 **Jupyter 시스템 자체의 문제**를 보고 싶을 때 확인합니다.

* **터미널/프롬프트 확인:** `jupyter notebook` 또는 `anaconda-navigator`를 실행했던 **검은색 터미널 창**을 보세요. 커널 연결 상태, 파일 저장 로그 등이 실시간으로 찍힙니다.
* **VS Code 사용 시:** 1. 하단 패널에서 **[출력(Output)]** 탭 클릭
    2. 오른쪽 드롭다운 메뉴에서 **[Jupyter]** 선택
    * 여기서 커널 시작, 익스텐션 연결 등의 상세 로그를 볼 수 있습니다.

---

## 4. 실시간 로그 확인 꿀팁 (파일로 저장)
로그 양이 방대하다면 화면에 띄우지 말고 파일로 기록하는 게 좋습니다.

```python
logging.basicConfig(
    filename='app.log', # 파일명
    filemode='a',       # 추가(append) 모드
    level=logging.DEBUG
)
```
이렇게 하면 셀을 실행해도 화면은 깨끗하고, 대신 `app.log` 파일에 모든 기록이 남습니다. 옆에 메모장으로 이 파일을 열어두면 편해요.

지금 어떤 종류의 로그(코드 에러 로그 vs 시스템 연결 로그)를 찾고 계신가요? 상황을 더 구체적으로 알려주시면 맞춤형 해결책을 드릴게요!

import csv
import json
import logging


def make_report(csv_path: str, json_path: str) -> int:
    
    try:

        # read file
        with open(csv_path,"r",encoding='utf-8') as f:
            reader = csv.DictReader(f)
            data = [{k: (v if v != '' else None) for k, v in row.items()} for row in reader] #코드 자동완성기능 사용

            #weighted sum    
            w_sum = [0] * len(data)


            for i in range(len(data)):
                if data[i]['중간'] is not None and data[i]['기말'] is not None and data[i]['과제'] is not None:
                    data[i]['중간'] = int(data[i]['중간']) #코드 자동완성기능 사용
                    data[i]['기말'] = int(data[i]['기말']) #코드 자동완성기능 사용
                    data[i]['과제'] = int(data[i]['과제']) #코드 자동완성기능 사용
                    w_sum[i] = 0.3*data[i]['중간'] + 0.5*data[i]['기말'] + 0.2*data[i]['과제']
                else:
                    w_sum[i] = None

                #determine grade (코드 자동완성 기능 사용)
                grades = []
                for score in w_sum:
                    if score is not None:
                        if score >= 90:
                            grades.append('A')
                        elif score >= 80:
                            grades.append('B')
                        elif score >= 70:
                            grades.append('C')
                        else:
                            grades.append('F')
                    else:
                        grades.append(None)

                #append score and grade to data (코드 자동완성 기능 사용)
                #”점수”: {”중간”: null, ”기말”: 82, ”과제”: 88} 형태로 출력 행바꿈 없이
                new_data = [{} for _ in range(len(data))]
                for i in range(len(data)):
                    new_data[i]['이름'] = data[i]['이름']
                    new_data[i]['학번'] = data[i]['학번']
                    new_data[i]['점수'] = {
                        '중간': data[i]['중간'],
                        '기말': data[i]['기말'],
                        '과제': data[i]['과제']
                    }
                    new_data[i]['평균'] = w_sum[i]
                    new_data[i]['등급'] = grades[i]
                    logging.info(f"{new_data[i]['이름']}의 평균 점수는 {new_data[i]['평균']}이고 등급은 {new_data[i]['등급']}입니다.") #코드 자동완성 기능 사용
                
                #write to json file(코드 자동완성 기능 사용)
                with open(json_path,"w",encoding='utf-8') as f:
                    json.dump(new_data,f,ensure_ascii=False,indent=4)
    
    
    except FileNotFoundError:
        logging.warning(f"파일 {csv_path}을 찾을 수 없습니다.")
        return 0
    except UnicodeDecodeError:
        logging.error(f"파일 {csv_path}을 읽는 중 인코딩 오류가 발생했습니다.")
        return 0
    return len(new_data)


make_report('scores.csv','report.json')
    

로그가 출력이 안되는데

작성하신 코드에서 로그가 출력되지 않는 이유는 크게 두 가지입니다.

1.  **로그 레벨 미설정:** 파이썬 `logging` 모듈의 기본 출력 레벨은 **`WARNING`**입니다. 코드에서 사용하신 `logging.info()`는 `WARNING`보다 낮은 단계라 무시됩니다.
2.  **커널 설정 문제:** `.ipynb`(주피터 노트북) 환경은 커널이 이미 로그 설정을 선점하고 있는 경우가 많아, 설정을 초기화해주어야 로그가 화면에 찍힙니다.

---

### 1. 로그 나오게 수정하기
코드 최상단(import 아래)에 이 설정을 추가하세요.

```python
# 기존 로그 설정을 초기화하고 INFO 레벨까지 보이게 설정
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
```

### 2. 점수 부분만 "한 줄로" 출력하기
앞서 원하셨던 **"점수 딕셔너리만 줄바꿈 없이 저장"**하는 기능을 코드에 반영했습니다. `json.dump`는 특정 부분만 줄바꿈을 안 하는 기능이 없어서, 저장 직전에 정규표현식으로 처리하는 방식을 넣었습니다.

---

### 최종 수정 코드 (로그 + 출력 형식 반영)

```python
import csv
import json
import logging
import re

# 로그 설정 초기화 (ipynb 필수)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def make_report(csv_path: str, json_path: str) -> int:
    try:
        with open(csv_path, "r", encoding='utf-8') as f:
            reader = csv.DictReader(f)
            data = [{k: (v if v != '' else None) for k, v in row.items()} for row in reader]

        new_data = []
        for row in data:
            # 점수 변환 및 가중치 합산
            mid = int(row['중간']) if row['중간'] is not None else None
            final = int(row['기말']) if row['기말'] is not None else None
            hw = int(row['과제']) if row['과제'] is not None else None
            
            w_sum = None
            grade = None
            
            if None not in (mid, final, hw):
                w_sum = 0.3 * mid + 0.5 * final + 0.2 * hw
                if w_sum >= 90: grade = 'A'
                elif w_sum >= 80: grade = 'B'
                elif w_sum >= 70: grade = 'C'
                else: grade = 'F'

            student_info = {
                "이름": row['이름'],
                "학번": row['학번'],
                "점수": {"중간": mid, "기말": final, "과제": hw},
                "평균": w_sum,
                "등급": grade
            }
            new_data.append(student_info)
            
            # 이제 로그가 찍힐 겁니다!
            logging.info(f"{student_info['이름']}의 평균 점수는 {student_info['평균']}이고 등급은 {student_info['등급']}입니다.")

        # 파일 쓰기: 점수 부분만 한 줄로 가공
        with open(json_path, "w", encoding='utf-8') as f:
            json_str = json.dumps(new_data, ensure_ascii=False, indent=4)
            # 정규표현식으로 "점수": { ... } 내부의 줄바꿈과 공백 제거
            compact_json = re.sub(r'("점수": \{)\s+(.*?)\s+(\})', 
                                  lambda m: m.group(1) + " ".join(m.group(2).split()) + m.group(3), 
                                  json_str, flags=re.DOTALL)
            f.write(compact_json)

        return len(new_data)

    except FileNotFoundError:
        logging.warning(f"파일 {csv_path}을 찾을 수 없습니다.")
        return 0
    except Exception as e:
        logging.error(f"오류 발생: {e}")
        return 0

make_report('scores.csv', 'report.json')
```

### 🛠️ 수정 포인트
1.  **`logging` 재설정:** 상단에 루프 돌면서 핸들러 삭제하는 코드를 넣어 `.ipynb`에서도 무조건 로그가 나오게 했습니다.
2.  **`re.sub` 추가:** `json.dumps` 이후에 `"점수": { ... }` 구간을 찾아 한 줄로 압축한 뒤 파일에 씁니다.
3.  **구조 개선:** `for`문 안에 `json.dump`가 있으면 파일이 계속 덮어씌워지므로, 루프가 끝난 뒤 마지막에 한 번만 저장하도록 위치를 옮겼습니다.

이제 셀을 실행하면 아래에 `INFO: 김언어의 평균...` 로그가 실시간으로 보일 거예요!

이렇게 출력되는데 어떻게 해결해

로그를 보니 지금 두 가지 문제가 섞여 있네요. 

1. **로그 중복 출력:** 셀을 실행할 때마다 로그가 누적되거나 중복해서 찍히고 있습니다.
2. **반복문 로직 오류:** 현재 코드 구조상 `new_data`를 만드는 루프 안에서 파일 쓰기(`with open...`)를 하거나, 리스트 생성 시점이 꼬여서 데이터가 중복 처리되고 있을 가능성이 큽니다.

가장 깔끔하게 **로그는 한 번만 딱 나오고, 파일 저장도 제대로 되는** 최종 수정 버전입니다.

---

### 수정된 코드 (로그 중복 & 로직 해결)

```python
import csv
import json
import logging
import re

# [해결 1] 로그 핸들러 중복 방지
logger = logging.getLogger()
if logger.hasHandlers():
    logger.handlers.clear()

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def make_report(csv_path: str, json_path: str) -> int:
    try:
        with open(csv_path, "r", encoding='utf-8') as f:
            reader = csv.DictReader(f)
            # 데이터를 먼저 리스트로 다 읽어옵니다.
            rows = list(reader)

        new_data = []
        for row in rows:
            # None 처리 (빈 문자열 체크)
            mid = int(row['중간']) if row.get('중간') else None
            final = int(row['기말']) if row.get('기말') else None
            hw = int(row['과제']) if row.get('과제') else None
            
            w_sum = None
            grade = None
            
            # 셋 중 하나라도 None이면 평균/등급은 None
            if None not in (mid, final, hw):
                w_sum = round(0.3 * mid + 0.5 * final + 0.2 * hw, 1)
                if w_sum >= 90: grade = 'A'
                elif w_sum >= 80: grade = 'B'
                elif w_sum >= 70: grade = 'C'
                else: grade = 'F'

            student_info = {
                "이름": row['이름'],
                "학번": row['학번'],
                "점수": {"중간": mid, "기말": final, "과제": hw},
                "평균": w_sum,
                "등급": grade
            }
            new_data.append(student_info)
            
            # 로그 출력
            logging.info(f"{student_info['이름']}의 평균 점수는 {student_info['평균']}이고 등급은 {student_info['등급']}입니다.")

        # [해결 2] 파일 쓰기는 반드시 루프 '밖에서' 딱 한 번만!
        with open(json_path, "w", encoding='utf-8') as f:
            json_str = json.dumps(new_data, ensure_ascii=False, indent=4)
            # 점수 부분만 한 줄로 합치기
            compact_json = re.sub(r'("점수": \{)\s+(.*?)\s+(\})', 
                                  lambda m: m.group(1) + " ".join(m.group(2).split()) + m.group(3), 
                                  json_str, flags=re.DOTALL)
            f.write(compact_json)

        return len(new_data)

    except FileNotFoundError:
        logging.warning(f"파일 {csv_path}을 찾을 수 없습니다.")
        return 0
    except Exception as e:
        logging.error(f"오류 발생: {e}")
        return 0

# 함수 실행
make_report('scores.csv', 'report.json')
```

---

### 무엇이 바뀌었나요?

1.  **`logger.handlers.clear()`**: 주피터 노트북에서 셀을 여러 번 실행하면 로그 설정이 겹쳐서 같은 문장이 여러 번 출력됩니다. 기존 설정을 싹 지우고 새로 잡도록 해서 중복 출력을 막았습니다.
2.  **데이터 처리 방식**: 기존 코드에서는 `for`문 안에 `new_data = [{} for _ in range(len(data))]` 같은 코드가 들어가 있어서 루프가 돌 때마다 리스트를 계속 새로 만들거나 덮어쓰고 있었을 겁니다. 이제 한 명씩 순서대로 `new_data`에 추가하도록 수정했습니다.
3.  **파일 저장 위치**: `with open(json_path, "w")...` 블록을 `for` 루프 **완전히 바깥으로** 뺐습니다. 이전에는 학생 한 명 처리할 때마다 파일을 계속 열어서 덮어쓰느라 비효율적이었을 거예요.
4.  **`round(..., 1)`**: 평균 점수가 `89.5000000004` 처럼 길게 나오는 걸 방지하기 위해 소수점 첫째 자리까지 반올림했습니다.

이제 실행해 보시면 학생 수만큼만 깔끔하게 로그가 찍힐 거예요!